### Part a)

 a) Implement the green/red-list watermark. I suggest doing so by making it a
HuggingFace LogitsProcessor (see the stub below) . Suggested defaults: γ = 0.5
(green-list fraction), δ = 2.0 (logit bias), k = 1 (hash context length). Generate
watermarked responses for the Dolly prompts and implement the detector that returns a z-score and p-value.

In [ ]:
import math
import hashlib
import torch
from transformers import LogitsProcessor, LogitsProcessorList


class GreenListWatermarkProcessor(LogitsProcessor):
    def __init__(self, vocab_size, gamma=0.5, delta=2.0, k=1):
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.delta = delta
        self.k = k
        self.green_size = int(gamma * vocab_size)

    def _seed_from_context(self, context_tokens):
        context_bytes = ",".join(map(str, context_tokens)).encode("utf-8")
        digest = hashlib.sha256(context_bytes).digest()
        return int.from_bytes(digest[:8], byteorder="big", signed=False)

    def _green_mask(self, context_tokens, device):
        seed = self._seed_from_context(context_tokens)

        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed)

        perm = torch.randperm(self.vocab_size, generator=generator)
        green_tokens = perm[: self.green_size].to(device)

        mask = torch.zeros(self.vocab_size, dtype=torch.bool, device=device)
        mask[green_tokens] = True

        return mask

    def __call__(self, input_ids, logits):
        # input_ids: (batch, seq_len)
        # logits: (batch, vocab_size)

        for batch_idx in range(input_ids.shape[0]):
            context = input_ids[batch_idx, -self.k :].tolist()
            green_mask = self._green_mask(context, logits.device)

            logits[batch_idx, green_mask] += self.delta

        return logits

In [ ]:
def detect_greenlist_watermark(
    text,
    tokenizer,
    vocab_size,
    gamma=0.5,
    k=1,
):
    token_ids = tokenizer(text, return_tensors="pt")["input_ids"][0].tolist()

    processor = GreenListWatermarkProcessor(
        vocab_size=vocab_size,
        gamma=gamma,
        delta=0.0,
        k=k,
    )

    green_count = 0
    total_count = 0

    for i in range(k, len(token_ids)):
        context = token_ids[i - k : i]
        token = token_ids[i]

        green_mask = processor._green_mask(context, device="cpu")

        if green_mask[token].item():
            green_count += 1

        total_count += 1

    expected = gamma * total_count
    variance = total_count * gamma * (1 - gamma)

    z_score = (green_count - expected) / math.sqrt(variance)

    # one-sided p-value: probability of seeing this many or more green tokens
    p_value = 0.5 * math.erfc(z_score / math.sqrt(2))

    return {
        "green_count": green_count,
        "total_count": total_count,
        "green_fraction": green_count / total_count if total_count > 0 else 0,
        "z_score": z_score,
        "p_value": p_value,
    }

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download


def download_model_weights(
    model_id: str = "Qwen/Qwen3-4B",
    model_path: str = "./Qwen3-4B",
    required_files: list[str] | None = None,
) -> Path:
    """
    Download model weights from Hugging Face if they are not already present locally.

    Args:
        model_id: Hugging Face model repository ID.
        model_path: Local directory where the model should be stored.
        required_files: Files used to check whether the model already exists.

    Returns:
        Path to the local model directory.
    """
    if required_files is None:
        required_files = [
            "config.json",
            "tokenizer.json",
        ]

    model_path = Path(model_path)

    model_exists = all(
        (model_path / file).exists()
        for file in required_files
    )

    if not model_exists:
        print(f"Model not found locally at {model_path}")
        print(f"Downloading {model_id} ...")

        snapshot_download(
            repo_id=model_id,
            local_dir=str(model_path),
            local_dir_use_symlinks=False,
        )

        print("Download complete.")
    else:
        print(f"Using existing local model at {model_path}")

    return model_path

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessorList
import torch


def load_model_weights(model_path: str = "./Qwen3-4B"):
    """
    Load tokenizer and model weights from a local model directory.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
        trust_remote_code=True,
    )

    if device == "cpu":
        model = model.to(device)

    model.eval()

    return model, tokenizer, device


def generate_response(
    prompt: str,
    model,
    tokenizer,
    watermarked: bool = True,
    gamma: float = 0.5,
    delta: float = 2.0,
    k: int = 1,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.95,
):
    """
    Generate a response from the model.

    If watermarked=True, applies the GreenListWatermarkProcessor.
    If watermarked=False, generates normally.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    generate_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": temperature,
        "top_p": top_p,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if watermarked:
        watermark_processor = GreenListWatermarkProcessor(
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            delta=delta,
            k=k,
        )

        generate_kwargs["logits_processor"] = LogitsProcessorList(
            [watermark_processor]
        )

    with torch.inference_mode():
        output_ids = model.generate(**generate_kwargs)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][prompt_len:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return response


def evaluate_watermark(
    text: str,
    tokenizer,
    vocab_size: int,
    gamma: float = 0.5,
    k: int = 1,
):
    """
    Evaluate whether a text contains the green-list watermark.
    """
    detection_result = detect_greenlist_watermark(
        text=text,
        tokenizer=tokenizer,
        vocab_size=vocab_size,
        gamma=gamma,
        k=k,
    )

    return detection_result

In [ ]:
from datasets import load_dataset
# Load Dolly dataset
dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train",
)

In [ ]:
MODEL_PATH = "./Qwen3-4B"
download_model_weights(
    model_id="Qwen/Qwen3-4B",
    model_path=MODEL_PATH,
)

# Load model + tokenizer
model, tokenizer, device = load_model_weights(MODEL_PATH)

# Prepare one prompt
example = dataset[0]

instruction = example["instruction"]
context = example["context"]

if context and len(context.strip()) > 0:
    prompt = f"""Instruction:
{instruction}

Context:
{context}

Response:"""
else:
    prompt = f"""Instruction:
{instruction}

Response:"""

print("PROMPT:")
print(prompt)

# Generate response
watermarked_response = generate_response(
    prompt=prompt,
    model=model,
    tokenizer=tokenizer,
    watermarked=True,
    gamma=0.5,
    delta=2.0,
    k=1,
)

print("\nWATERMARKED RESPONSE:")
print(watermarked_response)

# 5. Evaluate watermark
detection_result = evaluate_watermark(
    text=watermarked_response,
    tokenizer=tokenizer,
    vocab_size=model.config.vocab_size,
    gamma=0.5,
    k=1,
)

print("\nWATERMARK DETECTION:")
print(detection_result)

Using existing local model at Qwen3-4B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

PROMPT:
Instruction:
When did Virgin Australia start operating?

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

Response:

WATERMARKED RESPONSE:
 The answer is 31 August 2000.

But the user might be confused because of the name. The original name was Virgin Blue, but it was later renamed to Virgin Australia. The correct answer is 31 August 2000.

I need to make sure the user understands that even though the name changed, the start date remains the same.
The answer is 31 August 2000. Virgin Australia originally began oper

### Part b)

Evaluate robustness via a sanity check and paraphrase attack:

○ Sanity check. Run your detector on human responses and unwatermarked
Qwen3-4B responses; both should yield z-scores ≈ 0. Confirm watermarked
generations have meaningfully positive z-scores.

○ Paraphrase attack. Use an LLM to rewrite each watermarked response. Report
z-scores/p-values before and after.

#### b) - sanity check

In [ ]:
human_generated_str = "So I first went to Wikipedia and looked on the date when the company was founded. It is the year 2000, so I would assume the company Virgin Australia started to operate in that particular year. When reading through the article, this assumption is confirmed."


In [ ]:
# 8. Run detector on generated response
detection_result = detect_greenlist_watermark(
    text=human_generated_str,
    tokenizer=tokenizer,
    vocab_size=model.config.vocab_size,
    gamma=0.5,
    k=1,
)

print("\nWATERMARK DETECTION:")
print(detection_result)


WATERMARK DETECTION:
{'green_count': 28, 'total_count': 53, 'green_fraction': 0.5283018867924528, 'z_score': 0.4120816918460671, 'p_value': 0.34013977366722514}


In [ ]:
from datasets import load_dataset
import pandas as pd
import random


def build_dolly_prompt(example):
    """
    Build the same prompt format you already use for Dolly examples.
    """
    instruction = example["instruction"]
    context = example["context"]

    if context and len(context.strip()) > 0:
        prompt = f"""Instruction:
{instruction}

Context:
{context}

Response:"""
    else:
        prompt = f"""Instruction:
{instruction}

Response:"""

    return prompt


def evaluate_human_and_nonwatermarked_responses(
    model,
    tokenizer,
    dataset_name: str = "databricks/databricks-dolly-15k",
    split: str = "train",
    n_samples: int = 10,
    seed: int = 42,
    gamma: float = 0.5,
    k: int = 1,
    max_new_tokens: int = 256,
):
    """
    Samples Dolly examples, evaluates:
    1. Human-written Dolly responses
    2. Non-watermarked model-generated responses

    using the green-list watermark detector.
    """

    dataset = load_dataset(dataset_name, split=split)

    random.seed(seed)
    sample_indices = random.sample(range(len(dataset)), n_samples)

    results = []

    for idx in sample_indices:
        example = dataset[idx]

        prompt = build_dolly_prompt(example)
        human_response = example["response"]

        # Generate non-watermarked Qwen response
        nonwatermarked_response = generate_response(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            watermarked=False,
            gamma=gamma,
            k=k,
            max_new_tokens=max_new_tokens,
        )

        # Evaluate human response
        human_detection = evaluate_watermark(
            text=human_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        # Evaluate non-watermarked model response
        nonwatermarked_detection = evaluate_watermark(
            text=nonwatermarked_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        results.append({
            "dataset_index": idx,
            "instruction": example["instruction"],
            "context": example["context"],
            "human_response": human_response,
            "nonwatermarked_response": nonwatermarked_response,

            "human_green_fraction": human_detection["green_fraction"],
            "human_z_score": human_detection["z_score"],
            "human_p_value": human_detection["p_value"],

            "nonwatermarked_green_fraction": nonwatermarked_detection["green_fraction"],
            "nonwatermarked_z_score": nonwatermarked_detection["z_score"],
            "nonwatermarked_p_value": nonwatermarked_detection["p_value"],
        })

    return pd.DataFrame(results)

In [10]:
MODEL_PATH = "./Qwen3-4B"

# Optional, only if not already downloaded
download_model_weights(
    model_id="Qwen/Qwen3-4B",
    model_path=MODEL_PATH,
)

model, tokenizer, device = load_model_weights(MODEL_PATH)

sanity_df = evaluate_human_and_nonwatermarked_responses(
    model=model,
    tokenizer=tokenizer,
    n_samples=1,
    seed=42,
    gamma=0.5,
    k=1,
    max_new_tokens=256,
)

sanity_df[
    [
        "dataset_index",
        "human_z_score",
        "human_p_value",
        "nonwatermarked_z_score",
        "nonwatermarked_p_value",
    ]
]

Using existing local model at Qwen3-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

,dataset_index,human_z_score,human_p_value,nonwatermarked_z_score,nonwatermarked_p_value
0,10476,0.000000,0.500000,2.442275,0.007298
1,1824,-1.897367,0.971110,0.814092,0.207796
2,409,-0.577350,0.718149,0.438357,0.330564
3,12149,0.000000,0.500000,-0.187867,0.574510
4,4506,0.000000,0.500000,-1.064581,0.856467
5,4012,0.306987,0.379427,-0.688847,0.754540
6,3657,2.121320,0.016947,0.062622,0.475034
7,2286,-0.277350,0.609244,1.189826,0.117057
8,12066,2.921187,0.001744,-0.187867,0.574510
9,1679,1.047645,0.147401,-0.438357,0.669436


In [11]:
print("Human responses:")
print(sanity_df["human_z_score"].describe())

print("\nNon-watermarked model responses:")
print(sanity_df["nonwatermarked_z_score"].describe())

Human responses:
count    10.000000
mean      0.364507
std       1.370593
min      -1.897367
25%      -0.208013
50%       0.000000
75%       0.862481
max       2.921187
Name: human_z_score, dtype: float64

Non-watermarked model responses:
count    10.000000
mean      0.237965
std       1.029753
min      -1.064581
25%      -0.375735
50%      -0.062622
75%       0.720158
max       2.442275
Name: nonwatermarked_z_score, dtype: float64


#### b) - Paraphrase attack

### Part c)